In [1]:
include("CRD_STA.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using ProgressMeter
using DelimitedFiles

In [ ]:
num = 10
c = 0.52
A0,A1,A2,A3,A4 = KEB_SpatialMode.KEB_LST_ALL("Ekman.txt",199,0,0.137,116.3,0,2)
nep = PEP([A0,A1,A2,A3,A4]);
eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-10)
eigval

In [ ]:
pwd()

In [ ]:
cd("..")

In [3]:
##DIRECTLY CACULATE CUR
##initial
    for Tw = 1.06 : 0.02 : 1.2
    N_cheb = 149
    omega = 0
    Mr = 0.5
    gamma = 1.4
    sigma = 0.72
    global R = 500
    R_step = 0.2
    be0 = 0.14
    be1 = -0.1
    be_step = -0.0002
    c = 0.55
    num = 1
    Ma = Mr/R
    global u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,1,2)
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    num = 1
    
    global initial_i = []
    global initial_r = []
    global tempvec_i = [0 0 0 0]
    global tempvec_r = [0 0 0 0]
    global mode = 0
    writedlm("output_$Tw _ $Mr.dat",initial_i)
    writedlm("output_eig.dat",initial_r)

    for be = 0.17 :  0.25*be_step : 0.1

        A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb)
        nep = PEP([A0,A1,A2]); 
        eigval,eigvec = iar(nep,σ = c , neigs = 1 ,maxit = 500,tol=1e-10)
        point = filter(x ->  - 0.00015 < imag(x) < 0.00005, eigval)
        open("output_eig.dat", "a") do io
            write(io,"be=$be,eig=$eigval\n")
        end
        if point != []
            global initial_i = [omega R be imag(point)]
            global initial_r = [omega R be real(point)]
            break
        end
    end
    global total_r = initial_r
    global total_i = initial_i

# CACULATE

    for be = initial_r[1,3] +  be_step  :  be_step : be1

        A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,total_r[end,2],Ma,omega,be + 1 * be_step,N_cheb)
        nep = PEP([A0,A1,A2]); 
        eigval_1,eigvec_1 = iar(nep,σ = total_r[end,4]+im * total_i[end,4] , neigs = num ,maxit = 500,tol=1e-10)
        A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,total_r[end,2],Ma,omega,be - 3 * be_step,N_cheb)
        nep = PEP([A0,A1,A2]); 
        eigval_2,eigvec_2 = iar(nep,σ = total_r[end,4]+im * total_i[end,4] , neigs = num ,maxit = 500,tol=1e-10)
        if (imag(eigval_1[1]) < 0 && imag(eigval_2[1]) > 0) || (imag(eigval_1[1]) < 0 && imag(eigval_2[1]) < 0)

        global mode = 1

        elseif (imag(eigval_1[1]) > 0 && imag(eigval_2[1]) < 0) || (imag(eigval_1[1]) > 0 && imag(eigval_2[1]) > 0)

        global mode = 2

        end
        
        if length(total_r[:,1]) > 2 
           grad = abs(( total_r[end,2] - total_r[end - 1,2] ) / ( total_r[end,3] - total_r[end - 1,3] ))
        else
           grad = 0
        end
 
        if mode == 1 

            for R = total_r[end,2] - 0 * grad * be_step  : -R_step : 100

                A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb)
                nep = PEP([A0,A1,A2]); 
                eigval,eigvec = iar(nep,σ = total_r[end,4] + im * total_i[end,4] , neigs = num ,maxit = 500,tol=1e-10)


                global tempvec_i = [tempvec_i;[omega R be imag(eigval[1])]]
                global tempvec_r = [tempvec_r;[omega R be real(eigval[1])]]

                len = size(tempvec_i,1)

                open("output_$Tw _ $Mr.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if -0.0001 < imag(eigval[1]) < 0.0001

                    global total_r = [total_r;[tempvec_r[end,1] tempvec_r[end,2] tempvec_r[end,3] tempvec_r[end,4]]]
                    global total_i = [total_i;[tempvec_i[end,1] tempvec_i[end,2] tempvec_i[end,3] tempvec_i[end,4]]]

                    tempvec_i = [0 0 0 0]
                    tempvec_r = [0 0 0 0]
                    
                    break
                end
                
                if len > 30 && abs(tempvec_i[end,4]) > abs(tempvec_i[end-15,4])

                    mode = 2
                    tempvec_i = [0 0 0 0]
                    tempvec_r = [0 0 0 0]
                    break
                    
                end
            end        
        end


        if mode == 2

            for R = total_r[end,2] + 0 * grad * be_step: R_step : 600

                A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb)
                nep = PEP([A0,A1,A2]); 
                eigval,eigvec = iar(nep,σ = total_r[end,4]+im * total_i[end,4] , neigs = num ,maxit = 500,tol=1e-10)

                global tempvec_i = [tempvec_i;[omega R be imag(eigval[1])]]
                global tempvec_r = [tempvec_r;[omega R be real(eigval[1])]]

                len = size(tempvec_i,1)

                open("output_$Tw _ $Mr.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if -0.0001 < imag(eigval[1]) < 0.0001
                    global total_r = [total_r;[tempvec_r[end,1] tempvec_r[end,2] tempvec_r[end,3] tempvec_r[end,4]]]
                    global total_i = [total_i;[tempvec_i[end,1] tempvec_i[end,2] tempvec_i[end,3] tempvec_i[end,4]]]
                    global tempvec_i = [0 0 0 0]
                    global tempvec_r = [0 0 0 0]
                    break

                end

                if len > 30 && abs(tempvec_i[end,4]) > abs(tempvec_i[end-15,4])

                    global mode = 1
                    global tempvec_i = [0 0 0 0]
                    global tempvec_r = [0 0 0 0]
                    break
                    
                end
            end
        end

        if mode == 1

            for R = total_r[end,2] - 0 * grad * be_step: -R_step : 100

                if total_i[end,3] == be

                    break

                end 

                A0,A1,A2 = Spatial_mode(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb)
                nep = PEP([A0,A1,A2]); 
                eigval,eigvec = iar(nep,σ = total_r[end,4] + im * total_i[end,4] , neigs = num ,maxit = 500,tol=1e-10)

     
                global tempvec_i = [tempvec_i;[omega R be imag(eigval[1])]]
                global tempvec_r = [tempvec_r;[omega R be real(eigval[1])]]

                len = size(tempvec_i,1)

                open("output_$Tw _ $Mr.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if -0.0001 < imag(eigval[1]) < 0.0001
                    global total_r = [total_r;[tempvec_r[end,1] tempvec_r[end,2] tempvec_r[end,3] tempvec_r[end,4]]]
                    global total_i = [total_i;[tempvec_i[end,1] tempvec_i[end,2] tempvec_i[end,3] tempvec_i[end,4]]]
                    global tempvec_i = [0 0 0 0]
                    global tempvec_r = [0 0 0 0]
                    break
                end
                
                if len > 30 && abs(tempvec_i[end,4]) > abs(tempvec_i[end-15,4])

                global mode = 2
                global tempvec_i = [0 0 0 0]
                global tempvec_r = [0 0 0 0]
                    break
                    
                end

            end     
        end
 
        if total_i[end,2]>510 && size(total_i,1)>100
            break
        end
        # if total_i[end,2]>total_i[end-1,2]
        #     break
        # end

    end

    writedlm("Netwon_$Tw _$Mr _i.dat",total_i)
    writedlm("Netwon_$Tw _$Mr _r.dat",total_r)
    end


BoundsError: BoundsError: attempt to access 0-element Vector{Any} at index [1, 3]

In [ ]:
readdir()


In [ ]:
pwd()

In [ ]:
cd("work")

In [ ]:
data = ones(100,1)

In [ ]:
cd("omega")
for omega = 0:0.1:0.5
    mkdir("$omega")
    cd("$omega")
    for Tw = 0.1:0.1:1
        writedlm("output_$omega _ $Tw.dat",data)
    end
    cd("..")
end

In [ ]:
N_cheb = 199
Mr = 0.01
Tw = 1
gamma = 1.4
sigma = 0.72
omega = 0
be = 0.132
R_step = 0.5
be0 = 0.14
be1 = 0.16
be_step = 0.0001
num = 10
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,0,2)
H,T = T_ca(Mr,f,q,w0,gamma,Tw)
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
lam = - (2/3) * T
kappa = (1/sigma) * T
eig_all = []
R = 117
Ma = Mr/R
A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,0,2)
nep = PEP([A0,A1,A2]); 
eigval,eigvec = iar(nep,σ =  0.5, neigs = num ,maxit = 500,tol=1e-10)
eigval

In [ ]:
scatter(real(eigval),imag(eigval))

In [ ]:
plot(x,F)

In [ ]:
data = readdlm("Ekman.txt")

In [ ]:
plot(x,T)
# plot!(g[5,:])

In [ ]:
# data = readdlm("Netwon_0.9_0.3.dat")
data_all = total_i
writedlm("Netwon_$Tw _$Mr.dat",data_all)

In [ ]:
N = 199
θ = range(0,length=N+1,stop=pi)
x = reshape(-cos.(θ), N+1, 1)
c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
X = repeat(x, 1, N+1);
dX = X - X';
D = (c * (1 ./ c)') ./ (dX .+ I(N+1));
D = D - diagm(vec(sum(D, dims=2))); 
# for i=1:N+1
#     x[i] = 40 * x[i] .+ 40
# end
# D = (1/40) * D
# D2 = D^2
a = 0.4
b = 1
c = 1
for i=1:N+1
    D[i,:]=D[i,:].* (1-b*x[i]-(1-b)*(x[i]^3+c*(1-x[i]^2)))^2/(2a*(b .+ 3 * (1-b)*x[i]^2 - 2 * c * (1-b) * x[i]))
end
for i=1:N+1
    x[i] = a * (1+b*x[i]+(1-b)*(x[i]^3+c*(1-x[i]^2)))/(1-b*x[i]-(1-b)*(x[i]^3+c*(1-x[i]^2)))
    if x[i]>20
        x[i]=20
    end
end

scatter!(x)

In [ ]:
Tw = 1
N_cheb = 199
Mr = 0.3
gamma = 1.4
sigma = 0.72
omega = 0
global R = 500
R_step = 0.2
be0 = 0.14
be1 = 0.16
be_step = 0.0002
c = 0.09
num = 1
Ma = Mr/R
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,1,2)
H,T = T_ca(Mr,f,q,w0,gamma,Tw)
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
lam = - (2/3) * T
kappa = (1/sigma) * T
num = 1

In [ ]:
plot(q)

In [ ]:
data = readdlm("Netwon_1.1_0.3.dat")
for i = 3 : 1 : 623

    if data[i-1,1] == data[i,1]
        
        data = vcat(data[1:i-1,:],data[i+1:end,:])

    end
end

In [ ]:
Tw = 1 
N_cheb = 299
Mr = 2
gamma = 1.4
sigma = 0.72
omega = 0
global R = 500
R_step = 0.2
be0 = 0.14
be1 = 0.16
be_step = 0.0002
c = 0.09
num = 1
Ma = Mr/R
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,1,0)